# Agent: People (Phase 1, no MQTT)

This notebook implements a minimal local simulation for 50 people around Brøndby Stadium.

Phase 1 scope:
- One agent only
- Basic movement and state transitions
- No MQTT publish/subscribe
- Configuration loaded with `simulated_city.config.load_config()`

## Run checklist
- Run Cells 2-7 once to initialize and connect MQTT.
- Start Camera, Dashboard, and Control listeners before starting this publish loop.
- Run Cell 8 to publish person state and entry events.
- If you rerun Cell 8, interrupt old loops first to avoid duplicate publishers.

In [1]:
# Cell purpose: import required modules and load project configuration
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
import math
import random
import sys
import os
import time
from pathlib import Path
from typing import Literal

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if (src_dir / "simulated_city").exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

os.environ["SIMCITY_MQTT_PROFILES"] = "local"

from simulated_city.config import load_config
import simulated_city.mqtt as mqtt

cfg = load_config()
sim_cfg = cfg.simulation
mqtt_cfg = cfg.mqtt

print(f"Loaded config. MQTT base topic: {cfg.mqtt.base_topic}")
print(f"Simulation section present: {sim_cfg is not None}")

Loaded config. MQTT base topic: simulated-city/stadium
Simulation section present: True


In [2]:
# Cell purpose: define local data model for people state used in Phase 1
State = Literal["approaching_entry", "inside", "exiting", "exited"]

@dataclass
class PersonState:
    person_id: str
    name: str
    lat: float
    lon: float
    state: State
    color: Literal["white", "green", "red"]
    speed_kmh: float
    target_entry_id: str
    target_entry_lat: float
    target_entry_lon: float
    target_exit_id: str
    heading_east: float | None
    heading_north: float | None
    pause_until_tick: int
    spawn_tick: int

print("Defined PersonState dataclass for local simulation state.")

Defined PersonState dataclass for local simulation state.


In [8]:
# Cell purpose: set simulation constants and build entry/exit points from config with safe fallbacks
TOTAL_PEOPLE = 40
PEOPLE_PER_SPAWN = 10
SPAWN_DISTANCE_FROM_GATE_M = 50.0
TICK_SECONDS = 0.4
SPAWN_INTERVAL_S = 5.0
SPAWN_BATCH_SIZE = 1
TICKS = 1050
SPAWN_INTERVAL_TICKS = max(1, int(round(SPAWN_INTERVAL_S / TICK_SECONDS)))
ENTRY_PROXIMITY_M = 10.0
EXIT_REACHED_M = sim_cfg.exit_reached_m if sim_cfg else 8.0
INSIDE_REACHED_M = 8.0
TRUE_ALLOW_PROBABILITY = sim_cfg.true_allow_probability if sim_cfg else 0.80
SEED = sim_cfg.seed if sim_cfg and sim_cfg.seed is not None else 42
MIN_SPEED_KMH = 5.4
MAX_SPEED_KMH = 5.4
STEP_NOISE_M = 0.20

TURN_BLEND = 0.18
PAUSE_PROB_PER_TICK = 0.0
PAUSE_MIN_S = 1.0
PAUSE_MAX_S = 3.0
GATE_JITTER_MIN_M = 0.0
GATE_JITTER_MAX_M = 0.0
SPAWN_JITTER_MIN_M = 1.0
SPAWN_JITTER_MAX_M = 5.0

INSIDE_TARGET_LAT = 55.6489
INSIDE_TARGET_LON = 12.4186

DENIED_WAYPOINT_BY_ENTRY = {
    "E1": {"waypoint_id": "X_E1", "lat": 55.6484, "lon": 12.4165},
    "E2": {"waypoint_id": "X_E2", "lat": 55.6482, "lon": 12.4190},
    "E3": {"waypoint_id": "X_E3", "lat": 55.6499, "lon": 12.4217},
    "E4": {"waypoint_id": "X_E4", "lat": 55.6506, "lon": 12.4174},
}

# Fallback center near Brøndby Stadium if no simulation locations are configured
center_lat = sim_cfg.center_lat if sim_cfg else 55.6479
center_lon = sim_cfg.center_lon if sim_cfg else 12.4186

configured_locations = list(sim_cfg.locations) if sim_cfg and sim_cfg.locations else []

if configured_locations:
    entries = [
        {"entry_id": loc.location_id, "lat": loc.lat, "lon": loc.lon}
        for loc in configured_locations
    ]
else:
    entries = [
        {"entry_id": "E1", "lat": center_lat + 0.0014, "lon": center_lon - 0.0009},
        {"entry_id": "E2", "lat": center_lat + 0.0012, "lon": center_lon + 0.0011},
        {"entry_id": "E3", "lat": center_lat - 0.0012, "lon": center_lon + 0.0010},
        {"entry_id": "E4", "lat": center_lat - 0.0013, "lon": center_lon - 0.0011},
    ]

name_pool = [
    "CaptainWobble", "SirHonk", "ProfessorPickle", "BananaBeard", "NoodleWizard",
    "DancingPotato", "TurboMuffin", "GiggleGoblin", "SneakyPancake", "WaffleNinja",
    "BongoLlama", "CheeseRocket", "FunkyFerret", "MysticMeatball", "SassySausage",
    "JellyBeanBoss", "DiscoWalrus", "GoofyGiraffe", "PicklePirate", "ChuckleChicken"
]

print(f"Entries configured: {len(entries)}")
print(f"Using random seed: {SEED}")
print(f"Configured total people: {TOTAL_PEOPLE}")
print(f"Spawn points: 4, people per spawn: {PEOPLE_PER_SPAWN}")
print(f"Spawn distance from gate (m): {SPAWN_DISTANCE_FROM_GATE_M}")
print(f"Configured ticks: {TICKS}")
print(f"Configured tick interval (s): {TICK_SECONDS}")
print(f"Spawn interval (s): {SPAWN_INTERVAL_S} (batch size {SPAWN_BATCH_SIZE})")
print(f"Spawn jitter radius (m): {SPAWN_JITTER_MIN_M}-{SPAWN_JITTER_MAX_M}")
print(f"Configured walking speed (km/h): {MIN_SPEED_KMH:.1f}")
print(f"True allow probability: {TRUE_ALLOW_PROBABILITY:.2f}")
print(f"Inside target: lat={INSIDE_TARGET_LAT}, lon={INSIDE_TARGET_LON}")
for entry_id in sorted(DENIED_WAYPOINT_BY_ENTRY):
    denied_waypoint = DENIED_WAYPOINT_BY_ENTRY[entry_id]
    print(
        f"Denied target for {entry_id}: lat={denied_waypoint['lat']}, lon={denied_waypoint['lon']}"
    )

Entries configured: 4
Using random seed: 42
Configured total people: 40
Spawn points: 4, people per spawn: 10
Spawn distance from gate (m): 50.0
Configured ticks: 1050
Configured tick interval (s): 0.4
Spawn interval (s): 5.0 (batch size 1)
Spawn jitter radius (m): 1.0-5.0
Configured walking speed (km/h): 5.4
True allow probability: 0.80
Inside target: lat=55.6489, lon=12.4186
Denied target for E1: lat=55.6484, lon=12.4165
Denied target for E2: lat=55.6482, lon=12.419
Denied target for E3: lat=55.6499, lon=12.4217
Denied target for E4: lat=55.6506, lon=12.4174


In [4]:
# Cell purpose: define movement helpers (distance, heading, offsets, and nearest target lookups)
def meters_between(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    earth_radius_m = 6_371_000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return earth_radius_m * c

def normalize_vector(x: float, y: float) -> tuple[float, float]:
    norm = math.sqrt(x * x + y * y)
    if norm <= 1e-12:
        return 0.0, 0.0
    return x / norm, y / norm

def meters_to_latlon(lat: float, lon: float, east_m: float, north_m: float) -> tuple[float, float]:
    lat_delta = north_m / 111_111.0
    lon_scale = max(0.2, math.cos(math.radians(lat)))
    lon_delta = east_m / (111_111.0 * lon_scale)
    return lat + lat_delta, lon + lon_delta

def heading_to_target(lat: float, lon: float, target_lat: float, target_lon: float) -> tuple[float, float]:
    east_m = (target_lon - lon) * 111_111.0 * max(0.2, math.cos(math.radians(lat)))
    north_m = (target_lat - lat) * 111_111.0
    return normalize_vector(east_m, north_m)

def jitter_point(lat: float, lon: float, min_radius_m: float, max_radius_m: float, rng: random.Random) -> tuple[float, float]:
    radius_m = rng.uniform(min_radius_m, max_radius_m)
    angle_rad = rng.uniform(0.0, 2.0 * math.pi)
    east_m = radius_m * math.cos(angle_rad)
    north_m = radius_m * math.sin(angle_rad)
    return meters_to_latlon(lat, lon, east_m, north_m)

def nearest_point(lat: float, lon: float, points: list[dict], id_key: str) -> dict:
    return min(points, key=lambda point: (meters_between(lat, lon, point["lat"], point["lon"]), point[id_key]))

print("Movement helper functions are ready.")

Movement helper functions are ready.


In [9]:
# Cell purpose: initialize people from four gate-based spawn points (50m from each gate)
rng = random.Random(SEED)
people: list[PersonState] = []

ordered_entries = sorted(entries, key=lambda entry_item: entry_item["entry_id"])
if len(ordered_entries) < 4:
    raise ValueError("Expected at least 4 gate entries to build 4 spawn locations.")

spawn_specs = []
for gate in ordered_entries[:4]:
    gate_east_m = (gate["lon"] - center_lon) * 111_111.0 * max(0.2, math.cos(math.radians(gate["lat"])))
    gate_north_m = (gate["lat"] - center_lat) * 111_111.0
    outward_east, outward_north = normalize_vector(gate_east_m, gate_north_m)
    if abs(outward_east) < 1e-12 and abs(outward_north) < 1e-12:
        outward_east, outward_north = 0.0, 1.0

    spawn_lat, spawn_lon = meters_to_latlon(
        gate["lat"],
        gate["lon"],
        outward_east * SPAWN_DISTANCE_FROM_GATE_M,
        outward_north * SPAWN_DISTANCE_FROM_GATE_M,
    )

    denied_waypoint = DENIED_WAYPOINT_BY_ENTRY.get(gate["entry_id"])
    if denied_waypoint is None:
        raise ValueError(f"Missing denied waypoint for entry_id={gate['entry_id']}")

    spawn_specs.append(
        {
            "spawn_id": f"S_{gate['entry_id']}",
            "lat": spawn_lat,
            "lon": spawn_lon,
            "count": PEOPLE_PER_SPAWN,
            "target_entry_id": gate["entry_id"],
            "target_entry_lat": gate["lat"],
            "target_entry_lon": gate["lon"],
            "target_exit_id": denied_waypoint["waypoint_id"],
        }
    )

exit_waypoint_by_entry = {
    entry_id: waypoint.copy()
    for entry_id, waypoint in DENIED_WAYPOINT_BY_ENTRY.items()
}

ALLOWED_ENTRY_IDS = {entry_item["entry_id"] for entry_item in ordered_entries[:4]}
global_person_index = 0

for spawn_spec in spawn_specs:
    for local_spawn_index in range(spawn_spec["count"]):
        spawn_lat, spawn_lon = jitter_point(
            spawn_spec["lat"],
            spawn_spec["lon"],
            SPAWN_JITTER_MIN_M,
            SPAWN_JITTER_MAX_M,
            rng,
        )

        target_entry_lat, target_entry_lon = jitter_point(
            spawn_spec["target_entry_lat"],
            spawn_spec["target_entry_lon"],
            GATE_JITTER_MIN_M,
            GATE_JITTER_MAX_M,
            rng,
        )

        spawn_tick = local_spawn_index * SPAWN_INTERVAL_TICKS
        goofy_name = name_pool[global_person_index % len(name_pool)]
        person = PersonState(
            person_id=f"{goofy_name.lower()}-{global_person_index + 1:03d}",
            name=goofy_name,
            lat=spawn_lat,
            lon=spawn_lon,
            state="approaching_entry",
            color="white",
            speed_kmh=rng.uniform(MIN_SPEED_KMH, MAX_SPEED_KMH),
            target_entry_id=spawn_spec["target_entry_id"],
            target_entry_lat=target_entry_lat,
            target_entry_lon=target_entry_lon,
            target_exit_id=spawn_spec["target_exit_id"],
            heading_east=None,
            heading_north=None,
            pause_until_tick=0,
            spawn_tick=spawn_tick,
        )
        people.append(person)
        global_person_index += 1

if len(people) != TOTAL_PEOPLE:
    raise ValueError(f"Configured TOTAL_PEOPLE={TOTAL_PEOPLE}, but built {len(people)} people.")

print(f"Initialized {len(people)} people total.")
for spawn_spec in spawn_specs:
    print(
        f"{spawn_spec['spawn_id']} -> gate {spawn_spec['target_entry_id']} "
        f"at ({spawn_spec['lat']:.6f}, {spawn_spec['lon']:.6f})"
    )
print(f"Allowed entry IDs: {sorted(ALLOWED_ENTRY_IDS)}")
print(f"Initial white count: {sum(1 for person in people if person.color == 'white')}")
print("Spawn waves: all 4 spawn points release one person together every 5 seconds.")

Initialized 40 people total.
S_E1 -> gate E1 at (55.647971, 12.417459)
S_E2 -> gate E2 at (55.647767, 12.419717)
S_E3 -> gate E3 at (55.649356, 12.420370)
S_E4 -> gate E4 at (55.649451, 12.417136)
Allowed entry IDs: ['E1', 'E2', 'E3', 'E4']
Initial white count: 40
Spawn waves: all 4 spawn points release one person together every 5 seconds.


In [6]:
# Cell purpose: connect to MQTT broker and define Phase 3 publish topics
client_suffix = f"agent-people-phase3-{int(datetime.now(timezone.utc).timestamp())}"
client = mqtt.connect_mqtt(mqtt_cfg, client_id_suffix=client_suffix)

person_state_topic = f"{mqtt_cfg.base_topic}/person/state"
entry_event_topic = f"{mqtt_cfg.base_topic}/entry/event"

print(f"Connected to MQTT broker at {mqtt_cfg.host}:{mqtt_cfg.port}")
print(f"Publishing person state to: {person_state_topic}")
print(f"Publishing entry events to: {entry_event_topic}")
print(f"Client suffix: {client_suffix}")

Connected to MQTT broker at 127.0.0.1:1883
Publishing person state to: simulated-city/stadium/person/state
Publishing entry events to: simulated-city/stadium/entry/event
Client suffix: agent-people-phase3-1772698226


In [7]:
# Cell purpose: run simulation loop and publish Phase 3 MQTT messages (state + entry events)
simulation_log: list[dict] = []
inside_count = 0
state_publish_success = 0
state_publish_fail = 0
entry_event_publish_success = 0
entry_event_publish_fail = 0

for tick in range(TICKS):
    tick_ts = datetime.now(timezone.utc).isoformat()

    for person in people:
        if tick < person.spawn_tick:
            continue

        if person.state == "exited":
            continue

        if tick < person.pause_until_tick:
            continue

        if rng.random() < PAUSE_PROB_PER_TICK:
            pause_ticks = max(1, int(round(rng.uniform(PAUSE_MIN_S, PAUSE_MAX_S) / TICK_SECONDS)))
            person.pause_until_tick = tick + pause_ticks
            continue

        speed_mps = person.speed_kmh / 3.6
        step_forward_m = max(0.0, speed_mps * TICK_SECONDS)
        lateral_noise_m = rng.uniform(-STEP_NOISE_M, STEP_NOISE_M)

        if person.state == "approaching_entry":
            target_lat = person.target_entry_lat
            target_lon = person.target_entry_lon

            desired_east, desired_north = heading_to_target(person.lat, person.lon, target_lat, target_lon)
            if person.heading_east is None or person.heading_north is None:
                person.heading_east = desired_east
                person.heading_north = desired_north
            else:
                blended_east = (1.0 - TURN_BLEND) * person.heading_east + TURN_BLEND * desired_east
                blended_north = (1.0 - TURN_BLEND) * person.heading_north + TURN_BLEND * desired_north
                person.heading_east, person.heading_north = normalize_vector(blended_east, blended_north)

            if person.heading_east is None or person.heading_north is None:
                continue

            step_east_m = person.heading_east * step_forward_m - person.heading_north * lateral_noise_m
            step_north_m = person.heading_north * step_forward_m + person.heading_east * lateral_noise_m
            person.lat, person.lon = meters_to_latlon(person.lat, person.lon, step_east_m, step_north_m)

            distance_to_entry = meters_between(person.lat, person.lon, target_lat, target_lon)
            if distance_to_entry <= ENTRY_PROXIMITY_M:
                is_allowed = rng.random() < TRUE_ALLOW_PROBABILITY
                if is_allowed:
                    person.state = "inside"
                    person.color = "green"
                    person.heading_east = None
                    person.heading_north = None
                    inside_count += 1
                    event_type = "entered"
                else:
                    person.state = "exiting"
                    person.color = "red"
                    person.heading_east = None
                    person.heading_north = None
                    event_type = "denied"

                entry_event_payload = {
                    "person_id": person.person_id,
                    "timestamp": tick_ts,
                    "event_type": event_type,
                    "entry_id": person.target_entry_id,
                }
                if mqtt.publish_json_checked(client, entry_event_topic, entry_event_payload, qos=0):
                    entry_event_publish_success += 1
                else:
                    entry_event_publish_fail += 1

        elif person.state == "inside":
            target_lat = INSIDE_TARGET_LAT
            target_lon = INSIDE_TARGET_LON

            distance_to_inside = meters_between(person.lat, person.lon, target_lat, target_lon)
            if distance_to_inside > INSIDE_REACHED_M:
                desired_east, desired_north = heading_to_target(person.lat, person.lon, target_lat, target_lon)
                if person.heading_east is None or person.heading_north is None:
                    person.heading_east = desired_east
                    person.heading_north = desired_north
                else:
                    blended_east = (1.0 - TURN_BLEND) * person.heading_east + TURN_BLEND * desired_east
                    blended_north = (1.0 - TURN_BLEND) * person.heading_north + TURN_BLEND * desired_north
                    person.heading_east, person.heading_north = normalize_vector(blended_east, blended_north)

                step_east_m = person.heading_east * step_forward_m - person.heading_north * lateral_noise_m
                step_north_m = person.heading_north * step_forward_m + person.heading_east * lateral_noise_m
                person.lat, person.lon = meters_to_latlon(person.lat, person.lon, step_east_m, step_north_m)
            else:
                person.heading_east = None
                person.heading_north = None

        elif person.state == "exiting":
            exit_point = exit_waypoint_by_entry.get(person.target_entry_id)
            if exit_point is None:
                continue

            target_lat = exit_point["lat"]
            target_lon = exit_point["lon"]

            desired_east, desired_north = heading_to_target(person.lat, person.lon, target_lat, target_lon)
            if person.heading_east is None or person.heading_north is None:
                person.heading_east = desired_east
                person.heading_north = desired_north
            else:
                blended_east = (1.0 - TURN_BLEND) * person.heading_east + TURN_BLEND * desired_east
                blended_north = (1.0 - TURN_BLEND) * person.heading_north + TURN_BLEND * desired_north
                person.heading_east, person.heading_north = normalize_vector(blended_east, blended_north)

            if person.heading_east is None or person.heading_north is None:
                continue

            step_east_m = person.heading_east * step_forward_m - person.heading_north * lateral_noise_m
            step_north_m = person.heading_north * step_forward_m + person.heading_east * lateral_noise_m
            person.lat, person.lon = meters_to_latlon(person.lat, person.lon, step_east_m, step_north_m)

            distance_to_exit = meters_between(person.lat, person.lon, target_lat, target_lon)
            if distance_to_exit <= EXIT_REACHED_M:
                person.state = "exited"
                person.heading_east = None
                person.heading_north = None
                exit_event_payload = {
                    "person_id": person.person_id,
                    "timestamp": tick_ts,
                    "event_type": "exited",
                    "entry_id": person.target_entry_id,
                }
                if mqtt.publish_json_checked(client, entry_event_topic, exit_event_payload, qos=0):
                    entry_event_publish_success += 1
                else:
                    entry_event_publish_fail += 1

        person_state_payload = {
            "person_id": person.person_id,
            "name": person.name,
            "timestamp": tick_ts,
            "lat": person.lat,
            "lon": person.lon,
            "state": person.state,
            "color": person.color,
            "speed_kmh": person.speed_kmh,
            "target_entry_id": person.target_entry_id,
        }
        if mqtt.publish_json_checked(client, person_state_topic, person_state_payload, qos=0):
            state_publish_success += 1
        else:
            state_publish_fail += 1

    white_count = sum(1 for person in people if person.color == "white")
    green_count = sum(1 for person in people if person.color == "green")
    red_count = sum(1 for person in people if person.color == "red")

    simulation_log.append(
        {
            "tick": tick,
            "white": white_count,
            "green": green_count,
            "red": red_count,
            "inside_count": inside_count,
        }
    )

    time.sleep(TICK_SECONDS)

final_white = simulation_log[-1]["white"]
final_green = simulation_log[-1]["green"]
final_red = simulation_log[-1]["red"]
final_exited = sum(1 for person in people if person.state == "exited")

print("Phase 3 simulation + publishing complete.")
print(f"Final counts -> white={final_white}, green={final_green}, red={final_red}, exited={final_exited}, inside_count={inside_count}")
print(f"State publish results -> success={state_publish_success}, fail={state_publish_fail}")
print(f"Entry event publish results -> success={entry_event_publish_success}, fail={entry_event_publish_fail}")
print("Sample records (first 3 people):")
for person in people[:3]:
    print({
        "person_id": person.person_id,
        "name": person.name,
        "state": person.state,
        "color": person.color,
        "target_entry_id": person.target_entry_id,
    })

Phase 3 simulation + publishing complete.
Final counts -> white=0, green=33, red=7, exited=7, inside_count=33
State publish results -> success=34103, fail=0
Entry event publish results -> success=47, fail=0
Sample records (first 3 people):
{'person_id': 'p001', 'name': 'Alex', 'state': 'exited', 'color': 'red', 'target_entry_id': 'E1'}
{'person_id': 'p002', 'name': 'Maja', 'state': 'inside', 'color': 'green', 'target_entry_id': 'E1'}
{'person_id': 'p003', 'name': 'Jonas', 'state': 'exited', 'color': 'red', 'target_entry_id': 'E1'}
